In [ ]:
from openai import OpenAI

# Initialize the Groq client
free_client = OpenAI(
    api_key=grok_api_key ,  # Replace with your actual Groq key
    base_url="https://api.groq.com/openai/v1"
)

print("Groq Client successfully defined!")

Groq Client successfully dddddddddefined!


In [3]:
def llm(prompt):
    response = free_client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Groq's standard production free-tier model
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# Run the test again
prompt="Hello! Confirm if you are responding via Groq free API."
print(llm(prompt))

I'm an AI, and I'm responding via the Meta Llama AI model, which is a type of artificial intelligence designed to process and generate human-like text. I am not responding via the Groq free API. Is there anything else I can help you with?


In [4]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm excited you're interested in the course. However, I need a bit more information from you. Could you please tell me which course you're referring to and when it started? That way, I can help you determine if it's still possible to join and what steps you might need to take.


In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [6]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

You can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


# dataset.md V - below

In [7]:
#*The FAQ data is available as JSON from the DataTalks.Club website*
#Let's fetch it:

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    # The clean, standard way to write this line
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

In [ ]:
documents[1200]

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [ ]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

In [29]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
search_results = search(question)

In [31]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [32]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [ ]:
search_results

In [34]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [35]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [36]:
prompt = build_prompt(question, search_results)

In [ ]:
response = response = free_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    input=prompt
)

In [ ]:
response.output_text

In [ ]:
response.output[0].content[0].text